In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
SEED = 42
np.random.seed(SEED)

In [3]:
X_train = pd.read_csv("../data/X_train_enc.csv")
y_train = pd.read_csv("../data/y_train.csv")["label"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("\nOverall label distribution:")
print(y_train.value_counts())
print("\nOverall label distribution (proportion):")
print(y_train.value_counts(normalize=True))

X_train shape: (175341, 196)
y_train shape: (175341,)

Overall label distribution:
label
1    119341
0     56000
Name: count, dtype: int64

Overall label distribution (proportion):
label
1    0.680622
0    0.319378
Name: proportion, dtype: float64


IID split

In [4]:
num_clients = 10

indices = np.random.permutation(len(X_train))
split_indices = np.array_split(indices, num_clients)

clients_iid = {}

for i, idx in enumerate(split_indices):
    X_client = X_train.iloc[idx].reset_index(drop=True)
    y_client = y_train.iloc[idx].reset_index(drop=True)
    clients_iid[i] = (X_client, y_client)

print("IID client sizes:")
for i in clients_iid:
    print(f"Client {i}: {clients_iid[i][0].shape[0]}")

IID client sizes:
Client 0: 17535
Client 1: 17534
Client 2: 17534
Client 3: 17534
Client 4: 17534
Client 5: 17534
Client 6: 17534
Client 7: 17534
Client 8: 17534
Client 9: 17534


In [5]:
for i in clients_iid:
    print(f"\nClient {i} label distribution:")
    print(clients_iid[i][1].value_counts(normalize=True))


Client 0 label distribution:
label
1    0.680068
0    0.319932
Name: proportion, dtype: float64

Client 1 label distribution:
label
1    0.682959
0    0.317041
Name: proportion, dtype: float64

Client 2 label distribution:
label
1    0.680449
0    0.319551
Name: proportion, dtype: float64

Client 3 label distribution:
label
1    0.679024
0    0.320976
Name: proportion, dtype: float64

Client 4 label distribution:
label
1    0.684955
0    0.315045
Name: proportion, dtype: float64

Client 5 label distribution:
label
1    0.68256
0    0.31744
Name: proportion, dtype: float64

Client 6 label distribution:
label
1    0.678795
0    0.321205
Name: proportion, dtype: float64

Client 7 label distribution:
label
1    0.675602
0    0.324398
Name: proportion, dtype: float64

Client 8 label distribution:
label
1    0.684556
0    0.315444
Name: proportion, dtype: float64

Client 9 label distribution:
label
1    0.677256
0    0.322744
Name: proportion, dtype: float64


In [6]:
os.makedirs("../data/clients_iid", exist_ok=True)

for i in clients_iid:
    Xc, yc = clients_iid[i]
    Xc.to_csv(f"../data/clients_iid/client_{i}_X.csv", index=False)
    yc.to_frame("label").to_csv(f"../data/clients_iid/client_{i}_y.csv", index=False)

print("IID client files saved.")

IID client files saved.


Non-IID split

In [7]:
X_normal = X_train[y_train == 0].reset_index(drop=True)
y_normal = y_train[y_train == 0].reset_index(drop=True)

X_attack = X_train[y_train == 1].reset_index(drop=True)
y_attack = y_train[y_train == 1].reset_index(drop=True)

print("Normal samples:", len(X_normal))
print("Attack samples:", len(X_attack))

Normal samples: 56000
Attack samples: 119341


In [8]:
ratios = [
    (0.60, 0.40),  # Client 0
    (0.55, 0.45),  # Client 1
    (0.50, 0.50),  # Client 2
    (0.45, 0.55),  # Client 3
    (0.40, 0.60),  # Client 4
    (0.35, 0.65),  # Client 5
    (0.30, 0.70),  # Client 6
    (0.25, 0.75),  # Client 7
    (0.20, 0.80),  # Client 8
    (0.15, 0.85)   # Client 9
]

In [9]:
total_per_client = len(X_train) // 10

needed_normal = sum(int(total_per_client * r[0]) for r in ratios)
needed_attack = sum(total_per_client - int(total_per_client * r[0]) for r in ratios)

print("Available normal :", len(X_normal))
print("Needed normal    :", needed_normal)

print("Available attack :", len(X_attack))
print("Needed attack    :", needed_attack)

Available normal : 56000
Needed normal    : 65748
Available attack : 119341
Needed attack    : 109592


In [10]:
clients_noniid = {}

normal_pointer = 0
attack_pointer = 0

for i, (normal_ratio, attack_ratio) in enumerate(ratios):
    n_normal = int(total_per_client * normal_ratio)
    n_attack = total_per_client - n_normal

    Xc_normal = X_normal.iloc[normal_pointer:normal_pointer + n_normal]
    yc_normal = y_normal.iloc[normal_pointer:normal_pointer + n_normal]

    Xc_attack = X_attack.iloc[attack_pointer:attack_pointer + n_attack]
    yc_attack = y_attack.iloc[attack_pointer:attack_pointer + n_attack]

    normal_pointer += n_normal
    attack_pointer += n_attack

    Xc = pd.concat([Xc_normal, Xc_attack], axis=0).reset_index(drop=True)
    yc = pd.concat([yc_normal, yc_attack], axis=0).reset_index(drop=True)

    temp = Xc.copy()
    temp["label"] = yc.values
    temp = temp.sample(frac=1, random_state=SEED).reset_index(drop=True)

    yc = temp["label"]
    Xc = temp.drop(columns=["label"])

    clients_noniid[i] = (Xc, yc)

print("Non-IID client sizes:")
for i in clients_noniid:
    print(f"Client {i}: {clients_noniid[i][0].shape[0]}")

Non-IID client sizes:
Client 0: 17534
Client 1: 17534
Client 2: 17534
Client 3: 17534
Client 4: 17534
Client 5: 17534
Client 6: 17534
Client 7: 13922
Client 8: 14028
Client 9: 14904


In [11]:
for i in clients_noniid:
    print(f"\nClient {i} label distribution:")
    print(clients_noniid[i][1].value_counts(normalize=True))


Client 0 label distribution:
label
0    0.599977
1    0.400023
Name: proportion, dtype: float64

Client 1 label distribution:
label
0    0.54996
1    0.45004
Name: proportion, dtype: float64

Client 2 label distribution:
label
0    0.5
1    0.5
Name: proportion, dtype: float64

Client 3 label distribution:
label
1    0.550017
0    0.449983
Name: proportion, dtype: float64

Client 4 label distribution:
label
1    0.600034
0    0.399966
Name: proportion, dtype: float64

Client 5 label distribution:
label
1    0.650051
0    0.349949
Name: proportion, dtype: float64

Client 6 label distribution:
label
1    0.700011
0    0.299989
Name: proportion, dtype: float64

Client 7 label distribution:
label
1    0.94462
0    0.05538
Name: proportion, dtype: float64

Client 8 label distribution:
label
1    1.0
Name: proportion, dtype: float64

Client 9 label distribution:
label
1    1.0
Name: proportion, dtype: float64


In [12]:
os.makedirs("../data/clients_noniid", exist_ok=True)

for i in clients_noniid:
    Xc, yc = clients_noniid[i]
    Xc.to_csv(f"../data/clients_noniid/client_{i}_X.csv", index=False)
    yc.to_frame("label").to_csv(f"../data/clients_noniid/client_{i}_y.csv", index=False)

print("Non-IID client files saved.")

Non-IID client files saved.
